# Resolution and nonuniform weights in discrete transport

This notebook generates `fig:matching-resolution-and-weights`.  It uses the same canonical source and target geometry as `fig:matching-2d-cost-exponent`, and then changes only the feasible transport constraints.  The assignment problem is the special case of
$$
U(a,b)=\{P\geq 0 : P\mathbb 1=a,\; P^\top\mathbb 1=b\}
$$
where both marginals have equal masses and the same number of atoms.

In [ ]:
from pathlib import Path
import sys

for candidate in [Path.cwd(), Path.cwd() / "notebooks-figures", Path.cwd().parent / "notebooks-figures"]:
    if (candidate / "figure_style.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise RuntimeError("Could not locate figure_style.py")

import matplotlib.pyplot as plt
import numpy as np
import ot

from figure_style import (
    VIOLET,
    canonical_matching_clouds,
    draw_point_clouds,
    draw_transport_segments,
    figure_dir,
    padded_limits,
    remove_axes,
    save_pdf,
    setup_matplotlib,
)

setup_matplotlib()


## Three transport constraints on the same geometry

The balanced panel is the same source/target pair as the assignment figure.  The coarser-target panel samples fewer target atoms from the same three-mode mixture, so several source atoms feed the same target.  The weighted panel keeps the same target locations but changes the target masses, which forces visible splitting.

In [ ]:
alpha, beta_balanced, labels_balanced = canonical_matching_clouds(seed=2027, n_source=36)
_, beta_rectangular, labels_rectangular = canonical_matching_clouds(seed=2031, n_source=36, target_counts=(9, 8, 7))
beta_weighted = beta_balanced.copy()

n = len(alpha)
a_balanced = np.ones(n) / n
b_balanced = np.ones(len(beta_balanced)) / len(beta_balanced)
b_rectangular = np.ones(len(beta_rectangular)) / len(beta_rectangular)

cluster_masses = np.array([0.48, 0.32, 0.20])
counts = np.bincount(labels_balanced, minlength=3)
b_weighted = np.array([cluster_masses[k] / counts[k] for k in labels_balanced])
b_weighted = b_weighted / b_weighted.sum()

all_points = np.vstack([alpha, beta_balanced, beta_rectangular])
xlim, ylim = padded_limits(all_points, pad=0.18)


## Couplings and export

POT solves each finite transport linear program.  A positive entry $P_{ij}$ is drawn as a red-to-blue segment.  Segment width and opacity encode transported mass; blue marker areas encode the target marginal when it is not uniform over the same number of points.

In [ ]:
def squared_cost(x, y):
    return ot.dist(x, y, metric="euclidean") ** 2


def optimal_plan(x, y, a, b):
    return ot.emd(a, b, squared_cost(x, y), numItermax=200000)


plans = {
    "balanced": (beta_balanced, a_balanced, b_balanced, optimal_plan(alpha, beta_balanced, a_balanced, b_balanced)),
    "rectangular": (beta_rectangular, a_balanced, b_rectangular, optimal_plan(alpha, beta_rectangular, a_balanced, b_rectangular)),
    "weighted": (beta_weighted, a_balanced, b_weighted, optimal_plan(alpha, beta_weighted, a_balanced, b_weighted)),
}


In [ ]:
fig_name = "matching-resolution-and-weights"
out = figure_dir(fig_name)

for filename, (target, a, b, plan) in plans.items():
    pairs = [(i, j, float(plan[i, j])) for i in range(plan.shape[0]) for j in range(plan.shape[1]) if plan[i, j] > 1e-10]

    fig, ax = plt.subplots(figsize=(2.25, 2.25))
    draw_transport_segments(ax, alpha, target, pairs, color=VIOLET, min_width=0.16, max_width=1.45, alpha_scale=0.62, zorder=1)
    target_weights = None if filename == "balanced" else b
    draw_point_clouds(ax, alpha, target, target_weights=target_weights, base_size=13.0, zorder=3)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect("equal")
    remove_axes(ax)
    save_pdf(fig, out / f"{filename}.pdf", pad_inches=0.075)
    plt.close(fig)
